In [ ]:
metadata = {
    'sim_idx': [],
    'bush': [],
    'branch': [],
    'trial': [],
    'flex_modulus': [],
    'probe_height': [],
    'field_stiffness': [],
    'sim_stiffness': [],
    'force_error_10mm': [],
    'force_error_15mm': [],
    'force_error_20mm': [],
    'force_error_25mm': [],
    'force_error_30mm': [],
    'num_links': [],
    'discretization_RMSE': []}

BUSH_DICT = {1:1, 3:2, 5:3, 9:4, 14:5, 23:6}

USING_CLOUDCOMPARE = True
USING_SPLINE = False
OVERRIDE_DATA = False
RECORDING_DATA = True
time_now_str = str(datetime.date.today()) + '_' + str(datetime.datetime.now().hour) + '-' + str(datetime.datetime.now().minute)
results_folder = '../data/results/' + time_now_str
if RECORDING_DATA:
    os.mkdir(results_folder)
    print("Data is being recorded to folder: ", results_folder)
else:
    print("Data is NOT being recorded. Set RECORDING_DATA to True to save data to a folder.")
total_sim_idx = 0


# ----------------------------------------------------------------
# Start iterating  
# ----------------------------------------------------------------
for BUSH_NUM in [1, 3, 5, 9, 14, 23]:
    print ("Starting bush number: ", BUSH_NUM)
    print ("----------------------------------------")
    for BRANCH_NUM in [1, 2, 3]:
        print ("----------------------------------------")
        print ("Starting bush :", BUSH_NUM, " branch: ", BRANCH_NUM)
        print ("----------------------------------------")
     
        # ----------------------------------------------------------------
        # Load in the curve
        # ----------------------------------------------------------------
        if USING_CLOUDCOMPARE:

            # Handle the one exception where the branch bends back down and my segmenter doesn't build it correctly with the usual algorithm
            if BUSH_NUM == 9 and BRANCH_NUM == 2:
                angles_validated = False
                attempt = 1
                while not angles_validated:
                    segs_mm, RMSE = splineconverter.segment_curve_from_cloudcompare('../data/ccCurves/B' + str(BUSH_NUM) + '_branch' + str(BRANCH_NUM) + '_smoothpolyline_minbb.txt', 
                                                        make_plot=True, strictly_increasing=False)
                    segs_flipped = flip_segs_from_curve(segs_mm, print_output=False)
                    seg_angles = splineconverter.get_angles_between_segments(segs_flipped) 
                    if abs(max(seg_angles.min(), seg_angles.max(), key=abs))<np.pi/2:
                        print(f"Valid segmentation found!!")
                        angles_validated = True
                    else:
                        print(f"Invalid angles in segmentation attempt {attempt}; trying again.")
                        attempt += 1
                print(f"Angles between segments: {seg_angles}")
            
            else:
                segs_mm, RMSE = splineconverter.segment_curve_from_cloudcompare('../data/ccCurves/B' + str(BUSH_NUM) + '_branch' + str(BRANCH_NUM) + '_smoothpolyline_minbb.txt', 
                                                    make_plot=True, strictly_increasing=True)
            segs_cc = np.array(segs_mm) / 1000  # convert to meters from millimeters
            segs = segs_cc.copy()
            diamdata = pd.read_csv('../data/diameters/offset_branch_diameter_data.csv')
            
        elif USING_SPLINE:
            # Load the spline data from files and split it into segments
            # segs_cm = splineconverter.segment_spline_from_files('../data/splines/B3_take2/bush_1_cyl0_pts.json', 
                                                        # '../data/splines/B3_take2/bush_1_0.obj', make_plot=True)
            segs_cm, RMSE = splineconverter.segment_spline_from_obj_file('../data/splines/B3_take2/bush' + str(BUSH_NUM) + '_' + str(BRANCH_NUM) + '.obj', make_plot=True)
            # print(f"segs from spline converter: {segs_cm}")
            segs_spline = np.array(segs_cm) / 100  # convert to meters from centimeters
            segs = segs_spline.copy()
            diamdata = pd.read_csv('../data/diameters/spline_diameters.csv')
            
        diameter_data_df = pd.DataFrame(diamdata, columns=['Bush', 'Branch', 'Location', 'Diameter', 'Height'])
        
        # ----------------------------------------------------------------
        # Process the curve data into segments
        # ----------------------------------------------------------------
        # Correctly orient the segments for MuJoCo (they're upside down from the camera)
        segs_flipped = flip_segs_from_curve(segs, print_output=True)
        # splineconverter.plot_obj_file('../data/splines/B3_take2/bush_1_0.obj')
        # print(segs_flipped)
        
        # get the z-value of the midpoints of each segment
        midpoint_zs = get_midpoints(segs_flipped, print_output=True)

        # Get lengths of segments and angles between them (for setting up the MJCF model) 
        seg_lengths = splineconverter.get_segment_lengths(segs_flipped)
        seg_angles = splineconverter.get_angles_between_segments(segs_flipped)  

        # ----------------------------------------------------------------
        # Pull relevant data for this cane and trial
        # ----------------------------------------------------------------
        # Get the diameter data for this particular cane (bush, branch number)
        this_cane_data = diameter_data_df.query('Bush == ' + str(BUSH_NUM) + ' and Branch == ' + str(BRANCH_NUM))
        print(this_cane_data)

        # generate a linear fit for the diameters 
        this_cane_linearfit = np.polyfit(this_cane_data['Height'], this_cane_data['Diameter'], 1)
        def get_rad_at_height(height_in_m):
            diameter = this_cane_linearfit[0]*(height_in_m*1000) + this_cane_linearfit[1] #convert m to mm
            return diameter/2/1000.  # convert mm to m

        # Get the radii at the segment midpoints using the linear fit
        radii = get_rad_at_height(midpoint_zs)
        print("Radii at segment midpoints (m): ", radii)       
        
        for TRIAL_NUM in [1, 2, 3]: # loop through trials for this branch
 
            # Check to see if it's one of the exceptions we're skipping.. 
            if BUSH_NUM ==14:
                if BRANCH_NUM == 2 and TRIAL_NUM == 3:
                    print("Excluding trial 14/2/3 due to too many spikes")
                    continue
                elif BRANCH_NUM == 3 and TRIAL_NUM ==1:
                    print("Excluding trial 14/3/1 because camera data did not capture push point")
                    continue

            metadata['sim_idx'].append(total_sim_idx)
            total_sim_idx += 1
            metadata['bush'].append(BUSH_NUM)
            metadata['branch'].append(BRANCH_NUM)
            metadata['trial'].append(TRIAL_NUM)
            metadata['num_links'].append(len(segs_flipped)-1)
            metadata['discretization_RMSE'].append(RMSE)
            
            # ----------------------------------------------------------------
            # Load in the push data for this trial
            # ----------------------------------------------------------------
            # get the probe height for this trial (convert from mm to m)
            PROBE_HEIGHT = this_cane_data.query('Location == ' +str(TRIAL_NUM))['Height'].values[0]/1000  # convert mm to m
            print("Probe height (m): ", PROBE_HEIGHT)
            metadata['probe_height'].append(PROBE_HEIGHT)

            # import forces from csv
            filename = "bush_" + str(BUSH_DICT[BUSH_NUM]) + "_branch_" + str(BRANCH_NUM) + "_trial_" + str(TRIAL_NUM) + ".csv"
            pushdata = pd.read_csv('../data/imu_cropped_push_data/' + filename)
            TRIAL_LENGTH = pushdata.shape[0]
            print(f"Loaded push data from {filename} with {TRIAL_LENGTH} rows.")
            
            # ----------------------------------------------------------------
            # Build the MuJoCo model
            # ----------------------------------------------------------------
            # read in the base XML for the branch model (this has the world and the probe, but not the segments of the branch yet)
            with open('../urdf/branch_base.xml', 'r') as f:
                branch_xml = f.read()

            flex_modulus = 4.9e9  # Flexural modulus from average of 6 tested canes
            editor = CaneEditor(branch_xml, flex_mod=flex_modulus)
            metadata['flex_modulus'].append(flex_modulus)

            # BUILD THE CANE MODEL IN MJCF
            editor.build_branch_from_lengths(seg_lengths, radii, def_stiff=BRANCH_STIFFNESS, verbose=False)
            editor.offset_all_joints_in_direction('x', seg_angles[:,0])
            editor.offset_all_joints_in_direction('y', seg_angles[:,1])
            zero_pos = editor.get_zero_springref_pos()
            editor.total_length = segs_flipped[-1][2]
            editor.define_probe_site(PROBE_HEIGHT, zero_pos, verbose=False)
            editor.show_model_at_pos(zero_pos)
            # editor.show_model_at_pos_with_camera(zero_pos, 1)

            # save a picture of the mujoco rendering to images/mujocoRenders
            filename = '../images/mujocoRenders/' + time_now_str + '_bush' + str(BUSH_NUM) + '_branch'+ str(BRANCH_NUM) + '_trial' + str(TRIAL_NUM) + '_' + str(len(segs)-1) + 'segs' + '.pdf'
            if RECORDING_DATA:
                editor.save_picture_of_model(zero_pos, filename)

            
            # ----------------------------------------------------------------
            # Initialize the simulation
            # ----------------------------------------------------------------

            model = editor.model
            data = mujoco.MjData(model)
            data.qpos[:] = zero_pos
            mujoco.mj_forward(model, data)

            # simulation-specific settings
            model.opt.timestep = .00002
            model.opt.integrator = mujoco.mjtIntegrator.mjINT_IMPLICIT
            model.opt.solver = mujoco.mjtSolver.mjSOL_NEWTON 
            model.opt.tolerance = 1e-8

            FRAMERATE = 10      # Frames per second for rendering
            DURATION = 2 # TRIAL_LENGTH
            DATACAP_RATE = 20 # Hz
            init_warn_count = data.warning[mujoco.mjtWarning.mjWARN_BADQACC].number

            # variables to put data into lists for later plotting
            timevals = []
            posvals = []
            forcevals = []
            probevals = []
            # frames_stepped = []

            pre_control_timevals = []
            pre_control_forcevals = []
            pre_control_posvals = []
            pre_control_probeposes = []

            # -- Displacement controller setup --
            probe_site_id = model.site("probe_contact_site").id
            probe_body_id = int(model.site_bodyid[probe_site_id])
            probe_init_xpos = data.site_xpos[probe_site_id].copy()

            # -- Controller setup --
            pid = PID(Kp=150, Ki=22, Kd=10, setpoint=0)
            pid.output_limits = (-10, 50)  # force limits in Newtons

            pid.setpoint = 0 # mm, total displacement
            CTRL_POS_UPDATE_RATE = 2 # Hz, how often to update the control position
            last_force = 0.0
            print(f"Starting simulation of length {DURATION}s")
            
            with mujoco.Renderer(model, width=640, height=480) as renderer:
                while data.time < DURATION:
                     # -- measure current x-displacement of probe contact site --
                    current_disp_m = data.site_xpos[probe_site_id][0] - probe_init_xpos[0]

                    # -- PID: drive current displacement toward target --
                    force_x = last_force + pid(current_disp_m)

                    # -- apply force at probe contact site --
                    data.qfrc_applied[:] = 0
                    mujoco.mj_applyFT(model, data,
                                    np.array([force_x, 0.0, 0.0]),  # force [N]
                                    np.zeros(3),                     # torque
                                    data.site_xpos[probe_site_id],  # point of application (world frame)
                                    probe_body_id,
                                    data.qfrc_applied)

                    # step the simulation
                    mujoco.mj_step(model, data)
                    if data.warning[mujoco.mjtWarning.mjWARN_BADQACC].number > init_warn_count:
                        print("Acceleration is too damn high! Moving on...")
                        break
                    
                    # -- save data from sim  -- 
                    if len(timevals) < data.time * DATACAP_RATE:
                        timevals.append(data.time)
                        posvals.append(data.qpos.copy())
                        forcevals.append(force_x)
                        probevals.append(current_disp_m)
                    
                    # -- render the scene and save it --
                    # if len(frames_stepped) < data.time * FRAMERATE:
                    #     renderer.update_scene(data)
                    #     pixels = renderer.render()
                        # frames_stepped.append(pixels)

                    # update the control position at the specified rate
                    if int(data.time * CTRL_POS_UPDATE_RATE) > int((data.time - model.opt.timestep) * CTRL_POS_UPDATE_RATE):
                        pre_control_timevals.append(data.time)
                        pre_control_forcevals.append(force_x)
                        pre_control_posvals.append(data.qpos.copy())
                        pre_control_probeposes.append(current_disp_m)
                        last_force = force_x
                        print(f"Time: {data.time:.3f}s, Current Displacement: {current_disp_m*1000:.3f} mm, Force Applied: {force_x:.3f} N")
                        pid.setpoint += 0.0005  # increment by 0.5 mm (0.0005 m)                    
                        
            dpi=120 
            width=1200 
            height=400
            figsize=(width/dpi, height/dpi)
            fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

            probearray = np.array(probevals)*1000  # convert from m to mm for easier comparison to push data
            ax.plot(timevals, probearray, label='Simulation Probe Displacement', color='blue')
            ax.set_title(f'Bush {BUSH_NUM}, Branch {BRANCH_NUM}, Trial {TRIAL_NUM}')
            ax.set_xlabel('Time (s)')
            ax.set_ylabel('Probe Displacement (mm)')

            # Add a reference line that steps up by 0.001 every 1 second
            step_height = 0.5
            step_interval = .5
            step_times = np.arange(0, timevals[-1]+step_interval, step_interval)
            step_values = np.arange(0, step_height*len(step_times), step_height)
            # Interpolate to match the timevals for plotting as a step function
            step_ref = np.zeros_like(timevals)
            for i, t in enumerate(timevals):
                idx = np.searchsorted(step_times, t, side='right')
                step_ref[i] = step_values[idx-1] 
            ax.step(timevals, step_ref, where='post', linestyle='--', color='black', label='0.001 step every 1s')
            # ax.legend()
            ax.grid(True)
            
            # Convert lists to numpy arrays for easier manipulation
            pre_control_posvals_array = np.array(pre_control_posvals)*1000  # convert from m to mm for easier comparison to push data
            pre_control_timevals_array = np.array(pre_control_timevals)
            pre_control_forcevals_array = np.array(pre_control_forcevals)
            pre_control_probeposes_array = np.array(pre_control_probeposes)*1000  # convert from m to mm for easier comparison to push data

            sim_data_df = pd.DataFrame({
                'Time (s)': pre_control_timevals_array,
                'Probe (mm)': pre_control_probeposes_array,
                'Force (N)': pre_control_forcevals_array
            })

            pickle_filename = results_folder + '/sim_data_df_bush' + str(BUSH_NUM) + '_branch'+ str(BRANCH_NUM) + '_trial' + str(TRIAL_NUM) + '.pkl'
            if RECORDING_DATA:  
                with open(pickle_filename, 'wb') as f:
                    pickle.dump(sim_data_df, f)
         
            # generate a linear fit for the real force-displacement curve 
            fd_linearfit = np.polyfit(pushdata['actuator_displacement'], pushdata['Load (N)'], 1)
            def get_force_at_disp(disp_in_mm):
                force = fd_linearfit[0]*disp_in_mm + fd_linearfit[1]
                return force

            # generate a linear fit for the simulator probe force-displacement curve
            sim_fd_linearfit = np.polyfit(pre_control_probeposes_array, pre_control_forcevals_array, 1)
            # sim_fd_linearfit_first_10mm = np.polyfit(pre_control_posvals_array[0:10], pre_control_forcevals_array[0:10], 1)
            def get_sim_force_at_disp(disp_in_mm):
                force = sim_fd_linearfit[0]*disp_in_mm + sim_fd_linearfit[1]
                return force

            metadata['field_stiffness'].append(fd_linearfit[0])
            metadata['sim_stiffness'].append(sim_fd_linearfit[0])

            # I'm lame and bad at code so this is how we're doing this
            try:
                sim_force_10mm = pre_control_forcevals_array[np.where(pre_control_probeposes_array >= 10)[0]]
                real_disp_10mm_force = pushdata.query('`actuator_displacement` >= 10')['Load (N)'].values[0]
                metadata['force_error_10mm'].append(abs(sim_force_10mm - real_disp_10mm_force))
            except:
                metadata['force_error_10mm'].append(np.nan)
            try:
                sim_force_15mm = pre_control_forcevals_array[np.where(pre_control_probeposes_array >= 15)[0]]
                real_disp_15mm_force = pushdata.query('`actuator_displacement` >= 15')['Load (N)'].values[0]
                metadata['force_error_15mm'].append(abs(sim_force_15mm - real_disp_15mm_force))
            except:
                metadata['force_error_15mm'].append(np.nan)
            try:
                sim_force_20mm = pre_control_forcevals_array[np.where(pre_control_probeposes_array >= 20)[0][0]]
                real_disp_20mm_force = pushdata.query('`actuator_displacement` >= 20')['Load (N)'].values[0]
                metadata['force_error_20mm'].append(abs(sim_force_20mm - real_disp_20mm_force))
            except:
                metadata['force_error_20mm'].append(np.nan)
            try:
                sim_force_25mm = pre_control_forcevals_array[np.where(pre_control_probeposes_array >= 25)[0][0]]
                real_disp_25mm_force = pushdata.query('`actuator_displacement` >= 25')['Load (N)'].values[0]
                metadata['force_error_25mm'].append(abs(sim_force_25mm - real_disp_25mm_force))
            except:
                metadata['force_error_25mm'].append(np.nan)
            try:
                sim_force_30mm = pre_control_forcevals_array[np.where(pre_control_probeposes_array >= 30)[0][0]]
                real_disp_30mm_force = pushdata.query('`actuator_displacement` >= 30')['Load (N)'].values[0]
                metadata['force_error_30mm'].append(abs(sim_force_30mm - real_disp_30mm_force))
            except:
                metadata['force_error_30mm'].append(np.nan)
            
            fig = plt.figure()
            plt.plot(pushdata['actuator_displacement'], pushdata['Load (N)'], label='Measured Data', color='#377eb8')
            plt.plot(pushdata['actuator_displacement'], get_force_at_disp(pushdata['actuator_displacement']), label='Linear Fit', linestyle='--', color='#ff7f00')
            plt.plot(pre_control_probeposes_array, pre_control_forcevals_array, label='Simulator probe', linestyle='-', color='#4daf4a')
            plt.plot(pre_control_probeposes_array, get_sim_force_at_disp(pre_control_probeposes_array), label='Simulator Linear Fit', linestyle='--', color='#f781bf')
            plt.xlabel("Displacement (mm)")
            plt.ylabel("Load (N)")
            plt.legend()
            plt.grid()

            if OVERRIDE_DATA and RECORDING_DATA:
                filename = '../images/forcedisplacementplots/' + time_now_str + '_bush' + str(BUSH_NUM) + '_branch'+ str(BRANCH_NUM) + '_trial' + str(TRIAL_NUM) + '.pdf'
                # media.write_image(filename, gs, fmt='pdf')
                plt.savefig(filename, format="pdf")  
                pickle_filename = results_folder + '/metadata.pkl'
                with open(pickle_filename, 'wb') as f:
                    pickle.dump(metadata, f)
                
            plt.show()

# print(metadata)
# media.show_video(frames_stepped, fps=FRAMERATE)
if RECORDING_DATA:
    metadata_df = pd.DataFrame(metadata)
    metadata_df.to_csv('../data/results/metadata_' + str(datetime.date.today()) + '.csv', index=False)